# Rolling-origin evaluation of the promised-lead-time model

Evaluate the logistic regression selected in notebook 06 on monthly purchase cohorts from January through August 2018. The comparison uses the same regression, preprocessing and dates with and without `promised_days`. Both variants omit purchase month and year. August is included as a partial final cohort. Earlier experiments informed the choice of model and features, so these results are retrospective.

In [5]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, confusion_matrix, precision_recall_curve, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
split_dir = root / "data" / "processed" / "splits"
manifest = json.loads((split_dir / "split_manifest.json").read_text(encoding="utf-8"))

purchase = "order_purchase_timestamp"
delivery = "order_delivered_customer_date"
estimate = "order_estimated_delivery_date"
target = "is_delayed"
features = [c for c in manifest["feature_columns"] if c not in {"purchase_year", "purchase_month"}]
variants = {"without_promise": features, "with_promise": features + ["promised_days"]}
assert not set(features) & set(manifest["target_columns"] + manifest["metadata_columns"])

orders = pd.read_csv(
    root / "data" / "raw" / "olist_orders_dataset.csv",
    usecols=["order_id", delivery, estimate],
    parse_dates=[delivery, estimate],
)
cohort = pd.concat(
    [pd.read_parquet(root / split["path"]) for split in manifest["splits"].values()],
    ignore_index=True,
).merge(orders, on="order_id", how="left", validate="one_to_one")
assert cohort[[purchase, delivery, estimate, target]].notna().all().all()
assert cohort[delivery].ge(cohort[purchase]).all()
assert cohort[target].eq(cohort[delivery].dt.normalize().gt(cohort[estimate].dt.normalize())).all()

cohort["promised_days"] = (
    cohort[estimate].dt.normalize() - cohort[purchase].dt.normalize()
).dt.days

## Evaluation protocol

For a test month starting on date T, use the purchase cohort two months earlier to tune the alert threshold. Leave the month directly preceding T for delivery outcomes to mature. Fit on all earlier purchases with a recorded delivery before the threshold-tuning cohort begins. Tune each variant's threshold to maximize F1 using calibration orders delivered before T. Hold the fitted model and the threshold fixed across the entire test month.

For example, the June fold fits on purchases before April 1 with deliveries recorded before April 1, tunes on April purchases delivered before June 1, and scores June purchases. Test metrics use eventual observed outcomes. Folds have disjoint test orders, while expanding training periods share history.

In [6]:
def fit_and_evaluate(columns, train, calibration, test):
    model = make_pipeline(
        SimpleImputer(strategy="median", keep_empty_features=True),
        StandardScaler(),
        LogisticRegression(max_iter=2000, solver="lbfgs", random_state=42),
    )
    model.fit(train[columns].astype(float), train[target])
    calibration_scores = model.predict_proba(calibration[columns].astype(float))[:, 1]
    precision, recall, thresholds = precision_recall_curve(calibration[target], calibration_scores)
    f1 = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
    threshold = float(thresholds[np.argmax(f1)])

    scores = model.predict_proba(test[columns].astype(float))[:, 1]
    alerts = scores >= threshold
    tn, fp, fn, tp = confusion_matrix(test[target], alerts, labels=[0, 1]).ravel()
    return calibration_scores, scores, {
        "threshold": threshold,
        "ap": average_precision_score(test[target], scores),
        "roc_auc": roc_auc_score(test[target], scores),
        "precision": tp / max(tp + fp, 1),
        "recall": tp / (tp + fn),
        "f1": 2 * tp / (2 * tp + fp + fn),
        "alert_rate": (tp + fp) / len(test),
        "tp": tp, "fp": fp, "fn": fn,
    }


monthly_results = []
budget_inputs = {}
daily_inputs = {}
fold_details = []
for start in pd.date_range("2018-01-01", "2018-08-01", freq="MS"):
    end = start + pd.DateOffset(months=1)
    calibration_start = start - pd.DateOffset(months=2)
    calibration_end = calibration_start + pd.DateOffset(months=1)

    train = cohort.loc[(cohort[purchase] < calibration_start) & (cohort[delivery] < calibration_start)]
    calibration_cohort = cohort.loc[cohort[purchase].between(calibration_start, calibration_end, inclusive="left")]
    calibration = calibration_cohort.loc[calibration_cohort[delivery] < start]
    test = cohort.loc[cohort[purchase].between(start, end, inclusive="left")].sort_values(
        [purchase, "order_id"],
    )
    assert train[purchase].max() < calibration[purchase].min() < test[purchase].min()
    assert train[delivery].max() < calibration_start and calibration[delivery].max() < start
    assert all(frame[target].nunique() == 2 for frame in (train, calibration, test))

    month = start.strftime("%Y-%m")
    fold_details.append({
        "month": month, "calibration_month": calibration_start.strftime("%Y-%m"),
        "train_orders": len(train), "calibration_orders": len(calibration),
        "calibration_excluded": len(calibration_cohort) - len(calibration),
        "test_orders": len(test), "test_delays": test[target].sum(),
        "prevalence": test[target].mean(), "last_purchase_day": test[purchase].max().date(),
    })

    for variant, columns in variants.items():
        calibration_scores, test_scores, result = fit_and_evaluate(columns, train, calibration, test)
        monthly_results.append({"month": month, "variant": variant, **result})
        if variant == "with_promise":
            budget_inputs[month] = calibration_scores, test_scores, test[target].to_numpy()
            daily_inputs[month] = (
                test[[purchase, delivery, target]].assign(score=test_scores),
                calibration_scores,
                (calibration_end - calibration_start).days,
            )

folds = pd.DataFrame(fold_details).set_index("month")
metrics = pd.DataFrame(monthly_results).set_index(["month", "variant"])
display(folds.round(4))
display(metrics.round(4))

,calibration_month,train_orders,calibration_orders,calibration_excluded,test_orders,test_delays,prevalence,last_purchase_day
month,,,,,,,,
2018-01,2017-11,29322,6977,311,7069,403,0.0570,2018-01-31
2018-02,2017-12,33992,5444,69,6555,926,0.1413,2018-02-28
2018-03,2018-01,41197,6890,179,7003,1328,0.1896,2018-03-31
2018-04,2018-02,47794,6231,324,6798,306,0.0450,2018-04-30
2018-05,2018-03,53644,6816,187,6749,443,0.0656,2018-05-31
2018-06,2018-04,60468,6715,83,6096,71,0.0116,2018-06-30
2018-07,2018-05,68318,6726,23,6156,208,0.0338,2018-07-31
2018-08,2018-06,75429,6070,26,6351,393,0.0619,2018-08-29


threshold      ap  roc_auc  precision  recall  \
month   variant                                                          
2018-01 without_promise     0.0190  0.0749   0.5539     0.0583  0.9926   
        with_promise        0.0247  0.0847   0.5762     0.0633  0.6749   
2018-02 without_promise     0.0333  0.1731   0.5439     0.1612  0.4114   
        with_promise        0.0257  0.1967   0.6038     0.1651  0.7991   
2018-03 without_promise     0.0455  0.2654   0.6261     0.2682  0.3584   
        with_promise        0.0572  0.2872   0.6305     0.2653  0.4759   
2018-04 without_promise     0.0467  0.1258   0.7323     0.0672  0.8529   
        with_promise        0.0561  0.1724   0.7726     0.1044  0.7288   
2018-05 without_promise     0.0459  0.0921   0.6062     0.0826  0.7269   
        with_promise        0.0621  0.1478   0.7310     0.1248  0.6840   
2018-06 without_promise     0.0795  0.0250   0.6655     0.0343  0.3521   
        with_promise        0.1178  0.0748   0.7510     0.1373  0.1972   
2018-07 without_promise     0.0955  0.0459   0.5400     0.0618  0.3269   
        with_promise        0.1382  0.0752   0.6036     0.0515  0.3654   
2018-08 without_promise     0.1063  0.0453   0.3515     0.0242  0.0280   
        with_promise        0.1628  0.0711   0.5290     0.0547  0.2087   

                             f1  alert_rate   tp    fp   fn  
month   variant                                              
2018-01 without_promise  0.1101      0.9710  400  6464    3  
        with_promise     0.1157      0.6083  272  4028  131  
2018-02 without_promise  0.2317      0.3605  381  1982  545  
        with_promise     0.2736      0.6839  740  3743  186  
2018-03 without_promise  0.3068      0.2535  476  1299  852  
        with_promise     0.3407      0.3401  632  1750  696  
2018-04 without_promise  0.1246      0.5715  261  3624   45  
        with_promise     0.1826      0.3142  223  1913   83  
2018-05 without_promise  0.1483      0.5777  322  3577  121  
        with_promise     0.2111      0.3596  303  2124  140  
2018-06 without_promise  0.0626      0.1194   25   703   46  
        with_promise     0.1618      0.0167   14    88   57  
2018-07 without_promise  0.1039      0.1788   68  1033  140  
        with_promise     0.0903      0.2398   76  1400  132  
2018-08 without_promise  0.0260      0.0715   11   443  382  
        with_promise     0.0867      0.2360   82  1417  311

## Performance across months

AP is compared with each month's observed prevalence and with the paired regression. Report means across equally weighted months. For metrics at the chosen thresholds, also sum confusion counts to show how many alerts and discovered delays the eight folds would have produced.

In [7]:
ap_by_month = metrics["ap"].unstack()
ap_by_month["prevalence"] = folds["prevalence"]
ap_by_month["ap_gain"] = ap_by_month["with_promise"] - ap_by_month["without_promise"]
display(ap_by_month.round(4))

macro = metrics.groupby("variant")[["ap", "roc_auc", "f1"]].mean()
counts = metrics.groupby("variant")[["tp", "fp", "fn"]].sum()
counts["alert_rate"] = (counts["tp"] + counts["fp"]) / folds["test_orders"].sum()
counts["precision"] = counts["tp"] / (counts["tp"] + counts["fp"])
counts["recall"] = counts["tp"] / (counts["tp"] + counts["fn"])
display(macro.add_prefix("mean_").join(counts).round(4))
print("Mean monthly prevalence:", round(folds["prevalence"].mean(), 4))
print("Months with an AP gain:", int(ap_by_month["ap_gain"].gt(0).sum()), "/", len(ap_by_month))
print("Months above prevalence:", int(ap_by_month["with_promise"].gt(ap_by_month["prevalence"]).sum()), "/", len(ap_by_month))

variant,with_promise,without_promise,prevalence,ap_gain
month,,,,
2018-01,0.0847,0.0749,0.0570,0.0098
2018-02,0.1967,0.1731,0.1413,0.0237
2018-03,0.2872,0.2654,0.1896,0.0218
2018-04,0.1724,0.1258,0.0450,0.0466
2018-05,0.1478,0.0921,0.0656,0.0557
2018-06,0.0748,0.0250,0.0116,0.0498
2018-07,0.0752,0.0459,0.0338,0.0293
2018-08,0.0711,0.0453,0.0619,0.0258


,mean_ap,mean_roc_auc,mean_f1,tp,fp,fn,alert_rate,precision,recall
variant,,,,,,,,,
with_promise,0.1387,0.6497,0.1828,2342,16463,1736,0.3563,0.1245,0.5743
without_promise,0.1059,0.5774,0.1392,1944,19125,2134,0.3992,0.0923,0.4767


Mean monthly prevalence: 0.0757
Months with an AP gain: 8 / 8
Months above prevalence: 8 / 8


## Alert budget fixed before each window

Evaluate a 10% budget for the promised-lead-time model. Set the minimum alert score at the calibration cohort's 90th percentile. Then process the test month in purchase order, breaking timestamp ties by order ID. At each purchase, send an alert only if its score meets the minimum and the number of alerts so far is below 10% of observed purchases, rounded down. Unused capacity carries forward in the same month and expires at month-end. This enforces the cap without knowing the final order count or ranking future orders.

In [8]:
ORDERS_PER_ALERT = 10


def budget_alerts(scores, min_score):
    alerts = np.zeros(len(scores), dtype=bool)
    used = 0
    for index, score in enumerate(scores):
        capacity = (index + 1) // ORDERS_PER_ALERT
        if score >= min_score and used < capacity:
            alerts[index] = True
            used += 1
    assert np.all(alerts.cumsum() <= np.arange(1, len(scores) + 1) // ORDERS_PER_ALERT)
    return alerts


budget_rows = []
for month, (calibration_scores, test_scores, truth) in budget_inputs.items():
    min_score = float(np.quantile(calibration_scores, 1 - 1 / ORDERS_PER_ALERT))
    alerts = budget_alerts(test_scores, min_score)
    tn, fp, fn, tp = confusion_matrix(truth, alerts, labels=[0, 1]).ravel()
    budget_rows.append({
        "month": month, "min_score": min_score,
        "eligible_rate": (test_scores >= min_score).mean(),
        "capacity": len(truth) // ORDERS_PER_ALERT, "alerts": alerts.sum(),
        "alert_rate": alerts.mean(), "precision": tp / max(tp + fp, 1),
        "recall": tp / (tp + fn), "f1": 2 * tp / (2 * tp + fp + fn),
        "tp": tp, "fp": fp, "fn": fn,
    })

budget = pd.DataFrame(budget_rows).set_index("month")
f1_policy = metrics.xs("with_promise", level="variant")
policy_columns = ["alert_rate", "precision", "recall", "tp", "fp"]
display(budget[["min_score", "eligible_rate", "capacity", "alerts"]].round(4))
display(pd.concat(
    {"max_f1": f1_policy[policy_columns], "budget_10pct": budget[policy_columns]},
    axis=1,
).round(4))

summary = pd.DataFrame({
    "max_f1": f1_policy[["tp", "fp", "fn"]].sum(),
    "budget_10pct": budget[["tp", "fp", "fn"]].sum(),
}).T
summary["alerts"] = summary["tp"] + summary["fp"]
summary["alert_rate"] = summary["alerts"] / folds["test_orders"].sum()
summary["precision"] = summary["tp"] / summary["alerts"]
summary["recall"] = summary["tp"] / (summary["tp"] + summary["fn"])
summary["mean_monthly_f1"] = [f1_policy["f1"].mean(), budget["f1"].mean()]
display(summary.round(4))

,min_score,eligible_rate,capacity,alerts
month,,,,
2018-01,0.0493,0.0637,706,450
2018-02,0.0388,0.2171,655,655
2018-03,0.0531,0.4308,700,699
2018-04,0.0780,0.1308,679,678
2018-05,0.0929,0.1136,674,557
2018-06,0.0979,0.0413,609,252
2018-07,0.1475,0.2048,615,614
2018-08,0.1008,0.4711,635,634


max_f1                              budget_10pct            \
        alert_rate precision  recall   tp    fp   alert_rate precision   
month                                                                    
2018-01     0.6083    0.0633  0.6749  272  4028       0.0637    0.1311   
2018-02     0.6839    0.1651  0.7991  740  3743       0.0999    0.2015   
2018-03     0.3401    0.2653  0.4759  632  1750       0.0998    0.2704   
2018-04     0.3142    0.1044  0.7288  223  1913       0.0997    0.1490   
2018-05     0.3596    0.1248  0.6840  303  2124       0.0825    0.1831   
2018-06     0.0167    0.1373  0.1972   14    88       0.0413    0.0794   
2018-07     0.2398    0.0515  0.3654   76  1400       0.0997    0.0619   
2018-08     0.2360    0.0547  0.2087   82  1417       0.0998    0.0694   

                           
         recall   tp   fp  
month                      
2018-01  0.1464   59  391  
2018-02  0.1425  132  523  
2018-03  0.1423  189  510  
2018-04  0.3301  101  577  
2018-05  0.2302  102  455  
2018-06  0.2817   20  232  
2018-07  0.1827   38  576  
2018-08  0.1120   44  590

,tp,fp,fn,alerts,alert_rate,precision,recall,mean_monthly_f1
max_f1,2342,16463,1736,18805,0.3563,0.1245,0.5743,0.1828
budget_10pct,685,3854,3393,4539,0.0860,0.1509,0.1680,0.1504


## Fixed daily review capacity

Use 10, 20 and 30 reviews per calendar day as scenarios. Both policies use the same out-of-sample scores. Immediate alerts take the first qualifying orders in purchase order until capacity is exhausted. The score floor is set before the month, using the calibration quantile that corresponds to the daily capacity divided by average daily calibration volume. The daily queue ranks the day's purchases by score at the next midnight and reviews the highest-scoring orders still undelivered at that cutoff. It fills available slots from eligible orders. Ties follow purchase time and order ID. Capacity expires daily. The queue's added wait is measured from purchase to review. Delivered status at review time is assumed observable. Timestamps use the dataset's calendar without timezone conversion.

In [9]:
daily_rows = []
for month, (scored, calibration_scores, calibration_days) in daily_inputs.items():
    day = scored[purchase].dt.normalize()
    review_at = day + pd.Timedelta(days=1)
    open_at_review = scored[delivery] >= review_at
    for capacity in (10, 20, 30):
        fraction = min(capacity * calibration_days / len(calibration_scores), 1)
        min_score = np.quantile(calibration_scores, 1 - fraction)
        immediate = scored[scored["score"] >= min_score].groupby(day).head(capacity)
        queue = (
            scored[open_at_review].sort_values("score", ascending=False, kind="stable")
            .groupby(day).head(capacity)
        )
        for policy, alerted in [("immediate", immediate), ("daily_queue", queue)]:
            assert alerted.groupby(day).size().le(capacity).all()
            elapsed_hours = (
                (review_at - scored[purchase]).dt.total_seconds().div(3600)
                .loc[alerted.index]
            )
            daily_rows.append({
                "month": month, "capacity": capacity, "policy": policy,
                "alerts": len(alerted), "tp": int(alerted[target].sum()),
                "late_orders": int(scored[target].sum()),
                "wait_hours": elapsed_hours.sum() if policy == "daily_queue" else 0,
            })

daily_counts = pd.DataFrame(daily_rows)
daily_summary = daily_counts.groupby(["capacity", "policy"])[
    ["alerts", "tp", "late_orders", "wait_hours"]
].sum()
daily_summary["fp"] = daily_summary["alerts"] - daily_summary["tp"]
daily_summary["precision"] = daily_summary["tp"] / daily_summary["alerts"]
daily_summary["recall"] = daily_summary["tp"] / daily_summary["late_orders"]
daily_summary["mean_wait_hours"] = daily_summary["wait_hours"] / daily_summary["alerts"]
display(daily_summary.drop(columns="wait_hours").round(4))

monthly_20 = daily_counts.loc[daily_counts["capacity"].eq(20)].copy()
monthly_20["precision"] = monthly_20["tp"] / monthly_20["alerts"]
monthly_20["recall"] = monthly_20["tp"] / monthly_20["late_orders"]
display(monthly_20.pivot(
    index="month", columns="policy", values=["alerts", "tp", "precision", "recall"],
).round(4))

raw_orders = pd.read_csv(
    root / "data" / "raw" / "olist_orders_dataset.csv",
    usecols=["order_id", purchase, estimate],
    parse_dates=[purchase, estimate],
)
print("Raw orders:", len(raw_orders))
print("Duplicate order IDs:", raw_orders["order_id"].duplicated().sum())
print("Missing estimated dates:", raw_orders[estimate].isna().sum())
print("Estimated date before purchase day:",
      raw_orders[estimate].dt.normalize().lt(raw_orders[purchase].dt.normalize()).sum())

alerts   tp  late_orders    fp  precision  recall  \
capacity policy                                                           
10       daily_queue    2410  418         4078  1992     0.1734  0.1025   
         immediate      1847  292         4078  1555     0.1581  0.0716   
20       daily_queue    4811  725         4078  4086     0.1507  0.1778   
         immediate      3788  517         4078  3271     0.1365  0.1268   
30       daily_queue    7211  965         4078  6246     0.1338  0.2366   
         immediate      5664  703         4078  4961     0.1241  0.1724   

                      mean_wait_hours  
capacity policy                        
10       daily_queue           8.1688  
         immediate             0.0000  
20       daily_queue           8.2143  
         immediate             0.0000  
30       daily_queue           8.1633  
         immediate             0.0000

alerts                    tp             precision            \
policy  daily_queue immediate daily_queue immediate daily_queue immediate   
month                                                                       
2018-01       620.0     381.0        68.0      51.0      0.1097    0.1339   
2018-02       560.0     536.0       121.0      91.0      0.2161    0.1698   
2018-03       620.0     618.0       216.0     145.0      0.3484    0.2346   
2018-04       600.0     553.0       107.0      89.0      0.1783    0.1609   
2018-05       620.0     395.0        99.0      71.0      0.1597    0.1797   
2018-06       600.0     212.0        27.0      18.0      0.0450    0.0849   
2018-07       620.0     545.0        49.0      33.0      0.0790    0.0606   
2018-08       571.0     548.0        38.0      19.0      0.0665    0.0347   

             recall            
policy  daily_queue immediate  
month                          
2018-01      0.1687    0.1266  
2018-02      0.1307    0.0983  
2018-03      0.1627    0.1092  
2018-04      0.3497    0.2908  
2018-05      0.2235    0.1603  
2018-06      0.3803    0.2535  
2018-07      0.2356    0.1587  
2018-08      0.0967    0.0483

Raw orders: 99441
Duplicate order IDs: 0
Missing estimated dates: 0
Estimated date before purchase day: 0
